In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# ============================================
# 1. КЛАССЫ
# ============================================

class_names = ['Алевролит', 'Аргиллит', 'Глина', 'Переслаивание', 'Песчаник', 'Прочие', 'Углистые породы']

# ============================================
# 2. ОПРЕДЕЛЕНИЕ SIAMESE МОДЕЛЕЙ
# ============================================

class SiameseResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.resnet18(weights=None)
        in_features = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(in_features * 2, num_classes)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

class SiameseMobileNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.mobilenet_v3_large(weights=None)
        self.encoder.classifier = nn.Identity()
        for param in self.encoder.parameters():
            param.requires_grad = False
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            self.in_features = self.encoder(dummy).shape[1]
        self.classifier = nn.Linear(self.in_features * 2, num_classes)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

class SiameseEfficientNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = models.efficientnet_b3(weights=None)
        self.encoder.classifier = nn.Identity()
        for param in self.encoder.parameters():
            param.requires_grad = False
        with torch.no_grad():
            dummy = torch.randn(1, 3, 300, 300)
            self.in_features = self.encoder(dummy).shape[1]
        self.classifier = nn.Linear(self.in_features * 2, num_classes)
    
    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        return self.classifier(torch.cat((f1, f2), dim=1))

# ============================================
# 3. ЗАГРУЗКА МОДЕЛЕЙ
# ============================================

def load_siamese(model_class, path, num_classes=7):
    model = model_class(num_classes)
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

print("Загрузка моделей...")
model_resnet = load_siamese(SiameseResNet, "second/best_siamese_resnet.pth")
model_mobilenet = load_siamese(SiameseMobileNet, "second/best_siamese_mobilenet.pth")
model_efficientnet = load_siamese(SiameseEfficientNet, "second/best_siamese_efficientnet.pth")
print("✅ Модели загружены")

# ============================================
# 4. ТРАНСФОРМАЦИИ
# ============================================

transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_mobilenet = transform_resnet

transform_efficientnet = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 5. ПАПКА С MANUAL-ПАРАМИ (уточните путь)
# ============================================

manual_dir = Path("Digital_core_v5.2/manual_check")  # измените на свой путь

# Собираем все пары
pairs = []
for class_dir in manual_dir.iterdir():
    if not class_dir.is_dir():
        continue
    # Ищем файлы ds_* и uv_*
    ds_files = list(class_dir.glob("ds_*.jpg")) + list(class_dir.glob("*.jpg"))  # если нет префикса, нужно другое
    # Упростим: будем считать, что в папке класса лежат два файла: ds_*.jpg и uv_*.jpg
    ds_candidates = list(class_dir.glob("ds_*.jpg"))
    for ds_path in ds_candidates:
        uv_path = class_dir / ds_path.name.replace("ds_", "uv_")
        if uv_path.exists():
            pairs.append((class_dir.name, ds_path, uv_path))

if not pairs:
    # Если нет ds_*, попробуем просто взять все jpg и сгруппировать по имени без префикса
    print("Не найдены ds_* файлы, пробуем другой способ...")
    files = list(manual_dir.glob("*.jpg")) if manual_dir.is_dir() else []
    # Этот случай сложнее, поэтому лучше уточнить структуру.
    print("Уточните структуру папки second: есть ли подпапки классов? Как называются файлы?")
    exit()

print(f"Найдено пар в manual: {len(pairs)}")

# ============================================
# 6. ПРОВЕРКА ВСЕХ ТРЁХ МОДЕЛЕЙ
# ============================================

results = []
all_predictions = []  # для статистики

for true_label, ds_path, uv_path in pairs:
    ds_img = Image.open(ds_path).convert('RGB')
    uv_img = Image.open(uv_path).convert('RGB')
    
    # Показываем изображения
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(ds_img)
    axes[0].set_title(f"ДС: {true_label}")
    axes[0].axis('off')
    axes[1].imshow(uv_img)
    axes[1].set_title(f"УФ: {true_label}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    
    for model_name, model, transform in [
        ("ResNet18", model_resnet, transform_resnet),
        ("MobileNet", model_mobilenet, transform_mobilenet),
        ("EfficientNet", model_efficientnet, transform_efficientnet)
    ]:
        ds_tensor = transform(ds_img).unsqueeze(0).to(device)
        uv_tensor = transform(uv_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(ds_tensor, uv_tensor)
            probs = torch.nn.functional.softmax(output[0], dim=0)
            top3_probs, top3_indices = torch.topk(probs, 3)
            
            pred_class = top3_indices[0].item()
            confidence = top3_probs[0].item() * 100
            pred_label = class_names[pred_class]
            
            top3_str = ', '.join([f'{class_names[idx]} ({prob*100:.1f}%)' for idx, prob in zip(top3_indices, top3_probs)])
        
        results.append({
            'model': model_name,
            'true_label': true_label,
            'pred_label': pred_label,
            'confidence': confidence,
            'top3': top3_str,
            'ds_path': ds_path.name
        })
        
        status = "✅" if true_label == pred_label else "❌"
        print(f"{status} {model_name}: {true_label} → {pred_label} ({confidence:.1f}%)")
        print(f"   Топ-3: {top3_str}")
        print()

# ============================================
# 7. ДЕТАЛЬНАЯ СТАТИСТИКА ПО КАЖДОЙ МОДЕЛИ
# ============================================

for model_name in ["ResNet18", "MobileNet", "EfficientNet"]:
    model_results = [r for r in results if r['model'] == model_name]
    correct = sum(1 for r in model_results if r['true_label'] == r['pred_label'])
    total = len(model_results)
    print(f"\n{'='*60}")
    print(f"{model_name}: {correct}/{total} ({correct/total*100:.1f}%)")
    print(f"{'='*60}")
    
    for class_name in class_names:
        class_results = [r for r in model_results if r['true_label'] == class_name]
        if class_results:
            correct_class = sum(1 for r in class_results if r['true_label'] == r['pred_label'])
            total_class = len(class_results)
            print(f"{class_name}: {correct_class}/{total_class} ({correct_class/total_class*100:.1f}%)")